# Urban Flood Event Data Discovery

Survey satellite data availability for urban flood events across multiple cities.

**Sensors:**
- Sentinel-1 (C-band SAR, 10m)
- ALOS-2 PALSAR-2 (L-band SAR, 25m)
- Sentinel-2 (Optical, 10m)
- Landsat 8/9 (Optical, 30m)

In [1]:
import ee
import pandas as pd
from datetime import datetime, timedelta

# Initialize Earth Engine
ee.Authenticate()  # Uncomment if needed
ee.Initialize(project='urban-water-detection')

In [2]:
# Configuration
TEMPORAL_WINDOW_DAYS = 2  # Max days between optical and SAR for "coincident"
CLOUD_THRESHOLD = 30     # Max cloud % for usable optical imagery

# Urban mask from ESA WorldCover 2021
world_cover = ee.Image("ESA/WorldCover/v200/2021")
urban_mask = world_cover.eq(50)  # Class 50 = Built-up

In [3]:
def analyze_flood_event(name, lat, lon, buffer_km, flood_start, flood_end, 
                        event_type, country, notes=""):
    """
    Analyze satellite data availability for a flood event.
    Returns dict with counts and coincident pair information.
    """
    # Create AOI
    point = ee.Geometry.Point([lon, lat])
    aoi = point.buffer(buffer_km * 1000)
    
    # Urban fraction
    urban_frac = urban_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=100,
        maxPixels=1e9
    ).get('Map').getInfo()
    
    # Query Sentinel-1
    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(aoi)
          .filterDate(flood_start, flood_end)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .filter(ee.Filter.eq('instrumentMode', 'IW')))
    
    # Query ALOS-2
    alos2 = (ee.ImageCollection('JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR')
             .filterBounds(aoi)
             .filterDate(flood_start, flood_end))
    
    # Query Sentinel-2
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi)
          .filterDate(flood_start, flood_end))
    
    # Query Landsat 8/9
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(aoi)
          .filterDate(flood_start, flood_end))
    l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
          .filterBounds(aoi)
          .filterDate(flood_start, flood_end))
    landsat = l8.merge(l9)
    
    # Get counts
    s1_count = s1.size().getInfo()
    alos2_count = alos2.size().getInfo()
    s2_count = s2.size().getInfo()
    landsat_count = landsat.size().getInfo()
    
    # Usable optical (cloud < threshold)
    s2_usable = s2.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD)).size().getInfo()
    landsat_usable = landsat.filter(ee.Filter.lt('CLOUD_COVER', CLOUD_THRESHOLD)).size().getInfo()
    
    # Get detailed S1 info for coincident pair analysis
    s1_list = s1.toList(s1_count).getInfo() if s1_count > 0 else []
    s2_usable_list = (s2.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD))
                      .toList(s2_usable).getInfo() if s2_usable > 0 else [])
    landsat_usable_list = (landsat.filter(ee.Filter.lt('CLOUD_COVER', CLOUD_THRESHOLD))
                           .toList(landsat_usable).getInfo() if landsat_usable > 0 else [])
    
    # Find coincident pairs
    def find_pairs(sar_list, optical_list, window_days):
        pairs = []
        window_ms = window_days * 24 * 60 * 60 * 1000
        
        for sar in sar_list:
            sar_time = sar['properties']['system:time_start']
            sar_date = datetime.fromtimestamp(sar_time / 1000).strftime('%Y-%m-%d')
            
            matching = []
            for opt in optical_list:
                opt_time = opt['properties']['system:time_start']
                if abs(sar_time - opt_time) <= window_ms:
                    opt_date = datetime.fromtimestamp(opt_time / 1000).strftime('%Y-%m-%d')
                    cloud = opt['properties'].get('CLOUDY_PIXEL_PERCENTAGE', 
                            opt['properties'].get('CLOUD_COVER', 'N/A'))
                    matching.append({'date': opt_date, 'cloud': cloud})
            
            if matching:
                pairs.append({
                    'sar_date': sar_date,
                    'sar_orbit': sar['properties'].get('orbitProperties_pass', 'N/A'),
                    'sar_rel_orbit': sar['properties'].get('relativeOrbitNumber_start', 'N/A'),
                    'optical_matches': len(matching),
                    'best_optical': min(matching, key=lambda x: x['cloud'] if isinstance(x['cloud'], (int, float)) else 999)
                })
        
        return pairs
    
    all_optical = s2_usable_list + landsat_usable_list
    s1_pairs = find_pairs(s1_list, all_optical, TEMPORAL_WINDOW_DAYS)
    
    # ALOS-2 pairs
    alos2_list = alos2.toList(alos2_count).getInfo() if alos2_count > 0 else []
    alos2_pairs = find_pairs(alos2_list, all_optical, TEMPORAL_WINDOW_DAYS)
    
    # Print results
    print("=" * 60)
    print(f"EVENT: {name}")
    print(f"Type: {event_type} | Country: {country}")
    print(f"Period: {flood_start} to {flood_end}")
    print(f"Notes: {notes}")
    print(f"Urban fraction: {urban_frac:.3f}" if urban_frac else "Urban fraction: N/A")
    print("-" * 60)
    print("SAR Coverage:")
    print(f"  Sentinel-1 images: {s1_count}")
    print(f"  ALOS-2 images: {alos2_count}")
    print("Optical Coverage:")
    print(f"  Sentinel-2 total: {s2_count} | usable (<{CLOUD_THRESHOLD}% cloud): {s2_usable}")
    print(f"  Landsat 8/9 total: {landsat_count} | usable: {landsat_usable}")
    print(f"COINCIDENT PAIRS (within {TEMPORAL_WINDOW_DAYS} days):")
    print(f"  S1 with usable optical: {len(s1_pairs)}")
    print(f"  ALOS-2 with usable optical: {len(alos2_pairs)}")
    print("=" * 60)
    
    if s1_pairs:
        print("\nS1 Coincident Pair Details:")
        for p in s1_pairs:
            print(f"  SAR: {p['sar_date']} ({p['sar_orbit']}, orbit #{p['sar_rel_orbit']}) "
                  f"-> Optical: {p['best_optical']['date']} ({p['best_optical']['cloud']:.1f}% cloud)")
    
    if alos2_pairs:
        print("\nALOS-2 Coincident Pair Details:")
        for p in alos2_pairs:
            print(f"  SAR: {p['sar_date']} -> Optical: {p['best_optical']['date']} ({p['best_optical']['cloud']:.1f}% cloud)")
    
    return {
        'name': name,
        'type': event_type,
        'country': country,
        'urban_fraction': urban_frac,
        's1_count': s1_count,
        'alos2_count': alos2_count,
        's2_total': s2_count,
        's2_usable': s2_usable,
        'landsat_total': landsat_count,
        'landsat_usable': landsat_usable,
        's1_pairs': len(s1_pairs),
        'alos2_pairs': len(alos2_pairs),
        's1_pair_details': s1_pairs,
        'alos2_pair_details': alos2_pairs
    }

In [4]:
# Store results as we go
all_results = []

---
## North America

In [5]:
# Houston - Harvey 2017 (Baseline from paper)
houston = analyze_flood_event(
    name='Houston_Harvey_2017',
    lat=29.7604, lon=-95.3698,
    buffer_km=50,
    flood_start='2017-08-25',
    flood_end='2017-09-10',
    event_type='Hurricane',
    country='USA',
    notes='Baseline event from paper'
)
all_results.append(houston)

EVENT: Houston_Harvey_2017
Type: Hurricane | Country: USA
Period: 2017-08-25 to 2017-09-10
Notes: Baseline event from paper
Urban fraction: 0.223
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 5
  ALOS-2 images: 7
Optical Coverage:
  Sentinel-2 total: 38 | usable (<30% cloud): 17
  Landsat 8/9 total: 2 | usable: 1
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 5
  ALOS-2 with usable optical: 4

S1 Coincident Pair Details:
  SAR: 2017-08-28 (ASCENDING, orbit #34) -> Optical: 2017-08-30 (0.4% cloud)
  SAR: 2017-08-28 (ASCENDING, orbit #34) -> Optical: 2017-08-30 (0.4% cloud)
  SAR: 2017-09-04 (ASCENDING, orbit #136) -> Optical: 2017-09-04 (12.9% cloud)
  SAR: 2017-09-05 (DESCENDING, orbit #143) -> Optical: 2017-09-04 (12.9% cloud)
  SAR: 2017-08-30 (DESCENDING, orbit #143) -> Optical: 2017-08-30 (0.4% cloud)

ALOS-2 Coincident Pair Details:
  SAR: 2017-08-29 -> Optical: 2017-08-30 (0.4% cloud)
  SAR: 2017-08-29 -> Optical: 201

In [6]:
# Miami - Irma 2017
miami_irma = analyze_flood_event(
    name='Miami_Irma_2017',
    lat=25.7617, lon=-80.1918,
    buffer_km=30,
    flood_start='2017-09-09',
    flood_end='2017-09-20',
    event_type='Hurricane',
    country='USA',
    notes='Quick hurricane dispersal expected'
)
all_results.append(miami_irma)

EVENT: Miami_Irma_2017
Type: Hurricane | Country: USA
Period: 2017-09-09 to 2017-09-20
Notes: Quick hurricane dispersal expected
Urban fraction: 0.255
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 3
  ALOS-2 images: 2
Optical Coverage:
  Sentinel-2 total: 2 | usable (<30% cloud): 1
  Landsat 8/9 total: 0 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


In [7]:
# Midwest USA - Omaha 2019
midwest_2019 = analyze_flood_event(
    name='Midwest_USA_Omaha_2019',
    lat=41.2565, lon=-95.9345,
    buffer_km=40,
    flood_start='2019-03-13',
    flood_end='2019-04-15',
    event_type='Fluvial',
    country='USA',
    notes='Extended Missouri River flooding'
)
all_results.append(midwest_2019)

EVENT: Midwest_USA_Omaha_2019
Type: Fluvial | Country: USA
Period: 2019-03-13 to 2019-04-15
Notes: Extended Missouri River flooding
Urban fraction: 0.072
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 5
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 52 | usable (<30% cloud): 16
  Landsat 8/9 total: 7 | usable: 3
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 2
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2019-03-17 (ASCENDING, orbit #63) -> Optical: 2019-03-16 (0.0% cloud)
  SAR: 2019-03-29 (ASCENDING, orbit #63) -> Optical: 2019-03-31 (0.0% cloud)


In [8]:
# New Orleans - Ida 2021
new_orleans = analyze_flood_event(
    name='New_Orleans_Ida_2021',
    lat=29.9511, lon=-90.0715,
    buffer_km=35,
    flood_start='2021-08-29',
    flood_end='2021-09-10',
    event_type='Hurricane',
    country='USA',
    notes='Category 4, extensive urban flooding'
)
all_results.append(new_orleans)

EVENT: New_Orleans_Ida_2021
Type: Hurricane | Country: USA
Period: 2021-08-29 to 2021-09-10
Notes: Category 4, extensive urban flooding
Urban fraction: 0.086
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 1
  ALOS-2 images: 1
Optical Coverage:
  Sentinel-2 total: 20 | usable (<30% cloud): 9
  Landsat 8/9 total: 2 | usable: 2
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


In [9]:
# Fort Lauderdale 2023
fort_lauderdale = analyze_flood_event(
    name='Fort_Lauderdale_2023',
    lat=26.1224, lon=-80.1373,
    buffer_km=25,
    flood_start='2023-04-12',
    flood_end='2023-04-20',
    event_type='Flash_flood',
    country='USA',
    notes='April 2023 floods, 25 inches in 24hrs'
)
all_results.append(fort_lauderdale)

EVENT: Fort_Lauderdale_2023
Type: Flash_flood | Country: USA
Period: 2023-04-12 to 2023-04-20
Notes: April 2023 floods, 25 inches in 24hrs
Urban fraction: 0.319
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 2
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 4 | usable (<30% cloud): 4
  Landsat 8/9 total: 1 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


---
## Europe (Excellent mid-latitude coverage)

In [10]:
# Ahr Valley, Germany 2021
ahr_valley = analyze_flood_event(
    name='Ahr_Valley_Germany_2021',
    lat=50.4337, lon=7.1183,
    buffer_km=25,
    flood_start='2021-07-14',
    flood_end='2021-07-25',
    event_type='Fluvial',
    country='Germany',
    notes='Catastrophic flooding, well-documented'
)
all_results.append(ahr_valley)

EVENT: Ahr_Valley_Germany_2021
Type: Fluvial | Country: Germany
Period: 2021-07-14 to 2021-07-25
Notes: Catastrophic flooding, well-documented
Urban fraction: 0.059
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 8
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 16 | usable (<30% cloud): 11
  Landsat 8/9 total: 1 | usable: 1
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 6
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2021-07-18 (ASCENDING, orbit #88) -> Optical: 2021-07-18 (0.3% cloud)
  SAR: 2021-07-18 (ASCENDING, orbit #88) -> Optical: 2021-07-18 (0.3% cloud)
  SAR: 2021-07-22 (DESCENDING, orbit #139) -> Optical: 2021-07-21 (3.3% cloud)
  SAR: 2021-07-19 (ASCENDING, orbit #15) -> Optical: 2021-07-18 (0.3% cloud)
  SAR: 2021-07-21 (DESCENDING, orbit #37) -> Optical: 2021-07-21 (3.3% cloud)
  SAR: 2021-07-24 (ASCENDING, orbit #88) -> Optical: 2021-07-23 (8.9% cloud)


In [11]:
# Liege, Belgium 2021
liege = analyze_flood_event(
    name='Liege_Belgium_2021',
    lat=50.6326, lon=5.5797,
    buffer_km=20,
    flood_start='2021-07-14',
    flood_end='2021-07-25',
    event_type='Fluvial',
    country='Belgium',
    notes='Same event as Ahr Valley, simultaneous'
)
all_results.append(liege)

EVENT: Liege_Belgium_2021
Type: Fluvial | Country: Belgium
Period: 2021-07-14 to 2021-07-25
Notes: Same event as Ahr Valley, simultaneous
Urban fraction: 0.129
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 9
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 15 | usable (<30% cloud): 5
  Landsat 8/9 total: 2 | usable: 2
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 5
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2021-07-18 (ASCENDING, orbit #88) -> Optical: 2021-07-18 (0.1% cloud)
  SAR: 2021-07-22 (DESCENDING, orbit #139) -> Optical: 2021-07-21 (0.5% cloud)
  SAR: 2021-07-17 (ASCENDING, orbit #161) -> Optical: 2021-07-18 (0.1% cloud)
  SAR: 2021-07-17 (ASCENDING, orbit #161) -> Optical: 2021-07-18 (0.1% cloud)
  SAR: 2021-07-21 (DESCENDING, orbit #37) -> Optical: 2021-07-19 (0.1% cloud)


In [12]:
# Cologne, Germany 2021
cologne = analyze_flood_event(
    name='Cologne_Germany_2021',
    lat=50.9375, lon=6.9603,
    buffer_km=20,
    flood_start='2021-07-14',
    flood_end='2021-07-20',
    event_type='Fluvial',
    country='Germany',
    notes='Rhine flooding, large urban area'
)
all_results.append(cologne)

EVENT: Cologne_Germany_2021
Type: Fluvial | Country: Germany
Period: 2021-07-14 to 2021-07-20
Notes: Rhine flooding, large urban area
Urban fraction: 0.214
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 5
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 4 | usable (<30% cloud): 2
  Landsat 8/9 total: 0 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 2
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2021-07-18 (ASCENDING, orbit #88) -> Optical: 2021-07-18 (4.3% cloud)
  SAR: 2021-07-19 (ASCENDING, orbit #15) -> Optical: 2021-07-18 (4.3% cloud)


In [13]:
# Valencia, Spain 2024
valencia = analyze_flood_event(
    name='Valencia_Spain_2024',
    lat=39.4699, lon=-0.3763,
    buffer_km=25,
    flood_start='2024-10-29',
    flood_end='2024-11-10',
    event_type='Flash_flood',
    country='Spain',
    notes='October 2024 DANA floods'
)
all_results.append(valencia)

EVENT: Valencia_Spain_2024
Type: Flash_flood | Country: Spain
Period: 2024-10-29 to 2024-11-10
Notes: October 2024 DANA floods
Urban fraction: 0.118
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 7
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 12 | usable (<30% cloud): 2
  Landsat 8/9 total: 7 | usable: 3
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 7
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2024-10-31 (ASCENDING, orbit #103) -> Optical: 2024-10-30 (1.7% cloud)
  SAR: 2024-11-01 (DESCENDING, orbit #110) -> Optical: 2024-10-30 (1.7% cloud)
  SAR: 2024-11-01 (DESCENDING, orbit #110) -> Optical: 2024-10-30 (1.7% cloud)
  SAR: 2024-11-06 (DESCENDING, orbit #8) -> Optical: 2024-11-06 (15.6% cloud)
  SAR: 2024-11-06 (DESCENDING, orbit #8) -> Optical: 2024-11-06 (15.6% cloud)
  SAR: 2024-11-07 (ASCENDING, orbit #30) -> Optical: 2024-11-06 (15.6% cloud)
  SAR: 2024-11-07 (ASCENDING, orbit #30)

In [14]:
# Emilia-Romagna, Italy 2023
emilia_romagna = analyze_flood_event(
    name='Emilia_Romagna_Italy_2023',
    lat=44.4949, lon=11.3426,
    buffer_km=40,
    flood_start='2023-05-16',
    flood_end='2023-05-25',
    event_type='Fluvial',
    country='Italy',
    notes='Bologna region, extended flooding'
)
all_results.append(emilia_romagna)

EVENT: Emilia_Romagna_Italy_2023
Type: Fluvial | Country: Italy
Period: 2023-05-16 to 2023-05-25
Notes: Bologna region, extended flooding
Urban fraction: 0.062
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 4
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 8 | usable (<30% cloud): 4
  Landsat 8/9 total: 1 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 2
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2023-05-22 (DESCENDING, orbit #95) -> Optical: 2023-05-23 (6.4% cloud)
  SAR: 2023-05-23 (ASCENDING, orbit #117) -> Optical: 2023-05-23 (6.4% cloud)


In [15]:
# Paris - Seine 2016
paris = analyze_flood_event(
    name='Paris_Seine_2016',
    lat=48.8566, lon=2.3522,
    buffer_km=30,
    flood_start='2016-05-30',
    flood_end='2016-06-10',
    event_type='Fluvial',
    country='France',
    notes='Seine flooding, major urban center'
)
all_results.append(paris)

EVENT: Paris_Seine_2016
Type: Fluvial | Country: France
Period: 2016-05-30 to 2016-06-10
Notes: Seine flooding, major urban center
Urban fraction: 0.355
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 6
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 0 | usable (<30% cloud): 0
  Landsat 8/9 total: 1 | usable: 1
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 1
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2016-06-09 (DESCENDING, orbit #8) -> Optical: 2016-06-09 (15.6% cloud)


---
## East Asia

In [16]:
# Tokyo - Hagibis 2019
tokyo = analyze_flood_event(
    name='Tokyo_Hagibis_2019',
    lat=35.6762, lon=139.6503,
    buffer_km=40,
    flood_start='2019-10-12',
    flood_end='2019-10-20',
    event_type='Typhoon',
    country='Japan',
    notes='Typhoon Hagibis, rapid clearing typical'
)
all_results.append(tokyo)

EVENT: Tokyo_Hagibis_2019
Type: Typhoon | Country: Japan
Period: 2019-10-12 to 2019-10-20
Notes: Typhoon Hagibis, rapid clearing typical
Urban fraction: 0.560
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 7
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 2 | usable (<30% cloud): 0
  Landsat 8/9 total: 0 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


In [17]:
# Zhengzhou, China 2021
zhengzhou = analyze_flood_event(
    name='Zhengzhou_China_2021',
    lat=34.7472, lon=113.6249,
    buffer_km=30,
    flood_start='2021-07-17',
    flood_end='2021-07-28',
    event_type='Flash_flood',
    country='China',
    notes='1000-year rainfall event'
)
all_results.append(zhengzhou)

EVENT: Zhengzhou_China_2021
Type: Flash_flood | Country: China
Period: 2021-07-17 to 2021-07-28
Notes: 1000-year rainfall event
Urban fraction: 0.306
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 2
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 4 | usable (<30% cloud): 2
  Landsat 8/9 total: 0 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 2
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2021-07-27 (ASCENDING, orbit #40) -> Optical: 2021-07-25 (8.3% cloud)
  SAR: 2021-07-27 (ASCENDING, orbit #40) -> Optical: 2021-07-25 (8.3% cloud)


In [18]:
# Seoul 2022
seoul = analyze_flood_event(
    name='Seoul_2022',
    lat=37.5665, lon=126.9780,
    buffer_km=30,
    flood_start='2022-08-08',
    flood_end='2022-08-15',
    event_type='Flash_flood',
    country='South Korea',
    notes='August 2022 Seoul floods'
)
all_results.append(seoul)

EVENT: Seoul_2022
Type: Flash_flood | Country: South Korea
Period: 2022-08-08 to 2022-08-15
Notes: August 2022 Seoul floods
Urban fraction: 0.368
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 1
  ALOS-2 images: 1
Optical Coverage:
  Sentinel-2 total: 2 | usable (<30% cloud): 0
  Landsat 8/9 total: 0 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


In [19]:
# Osaka 2018
osaka = analyze_flood_event(
    name='Osaka_2018',
    lat=34.6937, lon=135.5023,
    buffer_km=30,
    flood_start='2018-07-05',
    flood_end='2018-07-15',
    event_type='Fluvial',
    country='Japan',
    notes='July 2018 Japan floods'
)
all_results.append(osaka)

EVENT: Osaka_2018
Type: Fluvial | Country: Japan
Period: 2018-07-05 to 2018-07-15
Notes: July 2018 Japan floods
Urban fraction: 0.457
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 2
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 0 | usable (<30% cloud): 0
  Landsat 8/9 total: 1 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


---
## Australia (Southern mid-latitude)

In [20]:
# Sydney 2022
sydney = analyze_flood_event(
    name='Sydney_2022',
    lat=-33.8688, lon=151.2093,
    buffer_km=40,
    flood_start='2022-02-23',
    flood_end='2022-03-15',
    event_type='Fluvial',
    country='Australia',
    notes='Extended Hawkesbury-Nepean flooding'
)
all_results.append(sydney)

EVENT: Sydney_2022
Type: Fluvial | Country: Australia
Period: 2022-02-23 to 2022-03-15
Notes: Extended Hawkesbury-Nepean flooding
Urban fraction: 0.155
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 10
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 20 | usable (<30% cloud): 1
  Landsat 8/9 total: 8 | usable: 1
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 1
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2022-03-10 (ASCENDING, orbit #9) -> Optical: 2022-03-09 (8.3% cloud)


In [21]:
# Brisbane 2022
brisbane = analyze_flood_event(
    name='Brisbane_2022',
    lat=-27.4698, lon=153.0251,
    buffer_km=35,
    flood_start='2022-02-26',
    flood_end='2022-03-10',
    event_type='Fluvial',
    country='Australia',
    notes='Same event system as Sydney'
)
all_results.append(brisbane)

EVENT: Brisbane_2022
Type: Fluvial | Country: Australia
Period: 2022-02-26 to 2022-03-10
Notes: Same event system as Sydney
Urban fraction: 0.187
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 3
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 14 | usable (<30% cloud): 5
  Landsat 8/9 total: 2 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 3
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2022-03-02 (DESCENDING, orbit #74) -> Optical: 2022-03-01 (22.5% cloud)
  SAR: 2022-03-02 (DESCENDING, orbit #74) -> Optical: 2022-03-01 (22.5% cloud)
  SAR: 2022-03-07 (DESCENDING, orbit #147) -> Optical: 2022-03-06 (4.5% cloud)


In [22]:
# Lismore, Australia 2022
lismore = analyze_flood_event(
    name='Lismore_Australia_2022',
    lat=-28.8133, lon=153.2750,
    buffer_km=20,
    flood_start='2022-02-28',
    flood_end='2022-03-15',
    event_type='Fluvial',
    country='Australia',
    notes='Record-breaking floods'
)
all_results.append(lismore)

EVENT: Lismore_Australia_2022
Type: Fluvial | Country: Australia
Period: 2022-02-28 to 2022-03-15
Notes: Record-breaking floods
Urban fraction: 0.010
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 4
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 18 | usable (<30% cloud): 0
  Landsat 8/9 total: 2 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


---
## South America

In [23]:
# Rio Grande do Sul, Brazil 2024
rio_grande_sul = analyze_flood_event(
    name='Rio_Grande_Sul_Brazil_2024',
    lat=-30.0346, lon=-51.2177,
    buffer_km=40,
    flood_start='2024-04-27',
    flood_end='2024-05-20',
    event_type='Fluvial',
    country='Brazil',
    notes='Porto Alegre area, extended flooding'
)
all_results.append(rio_grande_sul)

EVENT: Rio_Grande_Sul_Brazil_2024
Type: Fluvial | Country: Brazil
Period: 2024-04-27 to 2024-05-20
Notes: Porto Alegre area, extended flooding
Urban fraction: 0.098
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 3
  ALOS-2 images: 6
Optical Coverage:
  Sentinel-2 total: 35 | usable (<30% cloud): 5
  Landsat 8/9 total: 2 | usable: 1
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 1
  ALOS-2 with usable optical: 2

S1 Coincident Pair Details:
  SAR: 2024-05-08 (ASCENDING, orbit #163) -> Optical: 2024-05-08 (1.9% cloud)

ALOS-2 Coincident Pair Details:
  SAR: 2024-05-05 -> Optical: 2024-05-06 (0.0% cloud)
  SAR: 2024-05-06 -> Optical: 2024-05-06 (0.0% cloud)


---
## Arid Regions (Clear sky advantage)

In [24]:
# Dubai 2024
dubai = analyze_flood_event(
    name='Dubai_2024',
    lat=25.2048, lon=55.2708,
    buffer_km=30,
    flood_start='2024-04-15',
    flood_end='2024-04-20',
    event_type='Flash_flood',
    country='UAE',
    notes='Rare urban flood, excellent clear sky conditions'
)
all_results.append(dubai)

EVENT: Dubai_2024
Type: Flash_flood | Country: UAE
Period: 2024-04-15 to 2024-04-20
Notes: Rare urban flood, excellent clear sky conditions
Urban fraction: 0.227
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 1
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 6 | usable (<30% cloud): 3
  Landsat 8/9 total: 2 | usable: 2
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


In [25]:
# Derna, Libya 2023
derna = analyze_flood_event(
    name='Derna_Libya_2023',
    lat=32.7570, lon=22.6389,
    buffer_km=15,
    flood_start='2023-09-10',
    flood_end='2023-09-25',
    event_type='Dam_breach',
    country='Libya',
    notes='Catastrophic dam failure, clear arid conditions'
)
all_results.append(derna)

EVENT: Derna_Libya_2023
Type: Dam_breach | Country: Libya
Period: 2023-09-10 to 2023-09-25
Notes: Catastrophic dam failure, clear arid conditions
Urban fraction: 0.022
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 5
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 3 | usable (<30% cloud): 1
  Landsat 8/9 total: 5 | usable: 3
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 4
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2023-09-12 (ASCENDING, orbit #175) -> Optical: 2023-09-12 (9.4% cloud)
  SAR: 2023-09-13 (DESCENDING, orbit #7) -> Optical: 2023-09-12 (9.4% cloud)
  SAR: 2023-09-20 (DESCENDING, orbit #109) -> Optical: 2023-09-18 (2.7% cloud)
  SAR: 2023-09-20 (DESCENDING, orbit #109) -> Optical: 2023-09-18 (2.7% cloud)


In [26]:
# Jeddah 2022
jeddah = analyze_flood_event(
    name='Jeddah_2022',
    lat=21.4858, lon=39.1925,
    buffer_km=25,
    flood_start='2022-11-24',
    flood_end='2022-12-01',
    event_type='Flash_flood',
    country='Saudi Arabia',
    notes='Recent Jeddah flood with full sensor coverage'
)
all_results.append(jeddah)

EVENT: Jeddah_2022
Type: Flash_flood | Country: Saudi Arabia
Period: 2022-11-24 to 2022-12-01
Notes: Recent Jeddah flood with full sensor coverage
Urban fraction: 0.252
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 1
  ALOS-2 images: 0
Optical Coverage:
  Sentinel-2 total: 4 | usable (<30% cloud): 4
  Landsat 8/9 total: 1 | usable: 1
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 1
  ALOS-2 with usable optical: 0

S1 Coincident Pair Details:
  SAR: 2022-11-29 (ASCENDING, orbit #14) -> Optical: 2022-11-30 (26.6% cloud)


---
## South Asia (Monsoon - challenging but important)

In [27]:
# Kerala 2018
kerala = analyze_flood_event(
    name='Kerala_2018',
    lat=9.9312, lon=76.2673,
    buffer_km=40,
    flood_start='2018-08-08',
    flood_end='2018-08-30',
    event_type='Monsoon',
    country='India',
    notes='Extended flooding, may have post-monsoon clearing'
)
all_results.append(kerala)

EVENT: Kerala_2018
Type: Monsoon | Country: India
Period: 2018-08-08 to 2018-08-30
Notes: Extended flooding, may have post-monsoon clearing
Urban fraction: 0.041
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 6
  ALOS-2 images: 2
Optical Coverage:
  Sentinel-2 total: 0 | usable (<30% cloud): 0
  Landsat 8/9 total: 2 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


In [28]:
# Chennai 2015
chennai = analyze_flood_event(
    name='Chennai_2015',
    lat=13.0827, lon=80.2707,
    buffer_km=35,
    flood_start='2015-11-15',
    flood_end='2015-12-05',
    event_type='Fluvial',
    country='India',
    notes='December floods, post-monsoon'
)
all_results.append(chennai)

EVENT: Chennai_2015
Type: Fluvial | Country: India
Period: 2015-11-15 to 2015-12-05
Notes: December floods, post-monsoon
Urban fraction: 0.153
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 1
  ALOS-2 images: 2
Optical Coverage:
  Sentinel-2 total: 0 | usable (<30% cloud): 0
  Landsat 8/9 total: 0 | usable: 0
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 0
  ALOS-2 with usable optical: 0


In [29]:
# Pakistan Sindh 2022
pakistan = analyze_flood_event(
    name='Pakistan_Sindh_2022',
    lat=25.3960, lon=68.3578,
    buffer_km=50,
    flood_start='2022-08-01',
    flood_end='2022-09-30',
    event_type='Monsoon',
    country='Pakistan',
    notes='Massive extent, some clearing periods possible'
)
all_results.append(pakistan)

EVENT: Pakistan_Sindh_2022
Type: Monsoon | Country: Pakistan
Period: 2022-08-01 to 2022-09-30
Notes: Massive extent, some clearing periods possible
Urban fraction: 0.025
------------------------------------------------------------
SAR Coverage:
  Sentinel-1 images: 25
  ALOS-2 images: 14
Optical Coverage:
  Sentinel-2 total: 48 | usable (<30% cloud): 15
  Landsat 8/9 total: 24 | usable: 14
COINCIDENT PAIRS (within 2 days):
  S1 with usable optical: 13
  ALOS-2 with usable optical: 12

S1 Coincident Pair Details:
  SAR: 2022-08-03 (ASCENDING, orbit #42) -> Optical: 2022-08-04 (7.4% cloud)
  SAR: 2022-08-05 (DESCENDING, orbit #78) -> Optical: 2022-08-04 (7.4% cloud)
  SAR: 2022-08-27 (ASCENDING, orbit #42) -> Optical: 2022-08-28 (16.3% cloud)
  SAR: 2022-08-29 (DESCENDING, orbit #78) -> Optical: 2022-08-31 (7.8% cloud)
  SAR: 2022-09-03 (ASCENDING, orbit #144) -> Optical: 2022-09-05 (0.6% cloud)
  SAR: 2022-09-05 (DESCENDING, orbit #5) -> Optical: 2022-09-05 (0.6% cloud)
  SAR: 2022-09-0

---
## Summary Table

In [31]:
# Create summary dataframe
if all_results:
    df = pd.DataFrame(all_results)
    df['total_pairs'] = df['s1_pairs'] + df['alos2_pairs']
    
    # Grade each event
    def grade_event(row):
        if row['s1_pairs'] >= 3 and row['urban_fraction'] and row['urban_fraction'] > 0.15:
            return 'A'
        elif row['s1_pairs'] >= 1 and row['urban_fraction'] and row['urban_fraction'] > 0.10:
            return 'B'
        elif row['s1_count'] > 0:
            return 'C'
        else:
            return 'D'
    
    df['grade'] = df.apply(grade_event, axis=1)
    
    # Sort by grade then total pairs
    grade_order = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    df['grade_order'] = df['grade'].map(grade_order)
    df_sorted = df.sort_values(['grade_order', 'total_pairs'], ascending=[True, False])
    
    # Display summary
    display_cols = ['name', 'type', 'country', 'urban_fraction', 
                    's1_count', 's1_pairs', 'alos2_count', 'alos2_pairs', 'total_pairs', 'grade']
    print(df_sorted[display_cols].to_string(index=False))
    
    # Save to CSV
    from pathlib import Path
    output_path = Path(__file__).parent.parent / 'data' / 'discovery_summary.csv' if '__file__' in dir() else Path('discovery_summary.csv')
    df_sorted.to_csv(output_path, index=False)
    print("\nSaved to ../data/discovery_summary.csv")
else:
    print("No results yet. Run event cells above first.")

                      name        type      country  urban_fraction  s1_count  s1_pairs  alos2_count  alos2_pairs  total_pairs grade
       Houston_Harvey_2017   Hurricane          USA        0.222556         5         5            7            4            9     A
             Brisbane_2022     Fluvial    Australia        0.186618         3         3            0            0            3     A
       Valencia_Spain_2024 Flash_flood        Spain        0.117956         7         7            0            0            7     B
        Liege_Belgium_2021     Fluvial      Belgium        0.128811         9         5            0            0            5     B
      Cologne_Germany_2021     Fluvial      Germany        0.214369         5         2            0            0            2     B
      Zhengzhou_China_2021 Flash_flood        China        0.306378         2         2            0            0            2     B
          Paris_Seine_2016     Fluvial       France        0.354553  

---
## Grading Criteria

| Grade | Criteria |
|-------|----------|
| **A** | 3+ S1 images with usable optical pairs, >15% urban fraction |
| **B** | 1-2 S1 images with usable optical pairs, >10% urban fraction |
| **C** | SAR available but few/no usable optical pairs |
| **D** | No data or wrong time period (pre-Sentinel) |

---
## Next Steps

After discovery:
1. Select 3-5 Grade A/B events for training data
2. Document orbit/geometry decisions (ascending vs descending, incidence angle ranges)
3. Begin pseudo-label generation pipeline